In [1]:
# ライブラリのインポート
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import glob

## 1. Bright条件: ログ行数チェック（S01～S19）

In [ ]:
bright_results = []

for s in range(1, 20):
    subj = f"S{s:02d}"
    for seg in range(0, 10):
        csv_path = f"../../log/Bright/{subj}/{subj}_{seg}.csv"
        
        if not os.path.exists(csv_path):
            continue
        
        try:
            df = pd.read_csv(csv_path, encoding='utf-8-sig')
            n_rows = len(df)
            
            if n_rows != 48:
                bright_results.append({"Condition": "Bright", "Subject": subj, "Segment": seg, "Rows": n_rows})
        except Exception as e:
            bright_results.append({"Condition": "Bright", "Subject": subj, "Segment": seg, "Rows": f"エラー: {e}"})

if bright_results:
    df_bright = pd.DataFrame(bright_results)
    print(f"【Bright: 48行ではないログ】 {len(bright_results)}件")
    display(df_bright)
else:
    print("Bright: 全て48行です✓")

## 2. Dark条件: ログ行数チェック（S101～S119）

In [ ]:
dark_results = []

for s in range(101, 120):
    subj = f"S{s:03d}"
    for seg in range(0, 10):
        csv_path = f"../../log/Dark/{subj}/{subj}_{seg}.csv"
        
        if not os.path.exists(csv_path):
            continue
        
        try:
            df = pd.read_csv(csv_path, encoding='utf-8-sig')
            n_rows = len(df)
            
            if n_rows != 48:
                dark_results.append({"Condition": "Dark", "Subject": subj, "Segment": seg, "Rows": n_rows})
        except Exception as e:
            dark_results.append({"Condition": "Dark", "Subject": subj, "Segment": seg, "Rows": f"エラー: {e}"})

if dark_results:
    df_dark = pd.DataFrame(dark_results)
    print(f"【Dark: 48行ではないログ】 {len(dark_results)}件")
    display(df_dark)
else:
    print("Dark: 全て48行です✓")

## 3. Frame_120fps間隔チェック（Bright S01～S19, seg=0のみ）

In [ ]:
interval_results = []

for s in range(1, 20):
    subj = f"S{s:02d}"
    csv_path = f"../../log/Bright/{subj}/{subj}_0.csv"
    
    if not os.path.exists(csv_path):
        continue
    
    try:
        df = pd.read_csv(csv_path, encoding='utf-8-sig')
        
        if 'Frame_120fps' not in df.columns:
            interval_results.append({"Subject": subj, "Status": "Frame_120fps列なし", "Median": None, "Mean": None})
            continue
        
        frames = pd.to_numeric(df['Frame_120fps'], errors='coerce').dropna().values
        
        if len(frames) < 2:
            interval_results.append({"Subject": subj, "Status": f"データ不足 (n={len(frames)})", "Median": None, "Mean": None})
            continue
        
        intervals = np.diff(frames)
        median_interval = np.median(intervals)
        mean_interval = np.mean(intervals)
        
        # 300から±50以上外れているかチェック
        status = "★異常" if abs(median_interval - 300) > 50 else "正常"
        interval_results.append({
            "Subject": subj, 
            "Status": status, 
            "Median": round(median_interval, 1), 
            "Mean": round(mean_interval, 1)
        })
            
    except Exception as e:
        interval_results.append({"Subject": subj, "Status": f"エラー: {e}", "Median": None, "Mean": None})

df_intervals = pd.DataFrame(interval_results)
print("【Frame_120fps間隔チェック】")
display(df_intervals)

# 異常のみ表示
abnormal = df_intervals[df_intervals['Status'].str.contains('★', na=False)]
if len(abnormal) > 0:
    print(f"\n異常: {len(abnormal)}件")
    display(abnormal)

## 4. 総合サマリー

In [ ]:
print("="*60)
print("【総合サマリー】")
print("="*60)

total_abnormal = 0

if 'bright_results' in locals() and bright_results:
    print(f"✗ Bright: 48行ではないログ → {len(bright_results)}件")
    total_abnormal += len(bright_results)
else:
    print("✓ Bright: ログ行数OK")

if 'dark_results' in locals() and dark_results:
    print(f"✗ Dark: 48行ではないログ → {len(dark_results)}件")
    total_abnormal += len(dark_results)
else:
    print("✓ Dark: ログ行数OK")

if 'abnormal' in locals() and len(abnormal) > 0:
    print(f"✗ Frame間隔異常 → {len(abnormal)}件")
    total_abnormal += len(abnormal)
else:
    print("✓ Frame間隔OK")

print("="*60)
if total_abnormal == 0:
    print("🎉 全てのチェックをパスしました！")
else:
    print(f"⚠️  合計 {total_abnormal} 件の異常が見つかりました")

## 5. 画像確認：Pages と Task Windows の同時表示

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import glob

# 被験者とセグメント番号を指定
subject_id = "S01"  # 例: S01, S02, S03, S04
segment = 0         # 例: 0～9

condition = "Bright"  # または "Dark"

# ファイルパスを構築
pages_pattern = f"../../data/graphs/pages/{condition}/{subject_id}/{subject_id}_{segment}_with_emr/{subject_id}_{segment}_with_emr_page_*.png"
task_windows_pattern = f"../../data/graphs/task_windows/{condition}/{subject_id}/{subject_id}_{segment}_*.png"

pages_files = sorted(glob.glob(pages_pattern))
task_windows_files = sorted(glob.glob(task_windows_pattern))

print(f"【{condition} {subject_id} seg={segment}】")
print(f"Pages画像: {len(pages_files)}枚")
print(f"Task Windows画像: {len(task_windows_files)}枚")
print()

if pages_files or task_windows_files:
    # 全体のレイアウトを計算
    n_pages = min(len(pages_files), 3)  # 最大3枚
    n_task_windows = len(task_windows_files)
    
    # 行数を決定（Pages行 + Task Windows行）
    n_rows = (1 if n_pages > 0 else 0) + (1 if n_task_windows > 0 else 0)
    
    if n_rows > 0:
        fig = plt.figure(figsize=(24, 8 * n_rows))
        
        current_row = 1
        
        # Pages画像を上段に横並び
        if n_pages > 0:
            for i, img_path in enumerate(pages_files[:3]):
                ax = plt.subplot(n_rows, 3, i + 1)
                img = Image.open(img_path)
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(os.path.basename(img_path), fontsize=10)
            current_row += 1
        
        # Task Windows画像を下段に表示
        if n_task_windows > 0:
            for j, img_path in enumerate(task_windows_files):
                # 下段は3列全体を使用
                ax = plt.subplot(n_rows, 1, current_row + j)
                img = Image.open(img_path)
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(os.path.basename(img_path), fontsize=10)
        
        plt.suptitle(f"{condition} {subject_id} seg={segment}", fontsize=14, y=0.995)
        plt.tight_layout()
        
        # 画像を保存
        save_dir = f"../../data/graphs/page_task_combined/{condition}/{subject_id}"
        os.makedirs(save_dir, exist_ok=True)
        save_path = f"{save_dir}/{subject_id}_{segment}_combined.png"
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ 保存: {save_path}\n")
        
        plt.show()
        plt.close()
else:
    print("⚠️  画像が見つかりません")

### 複数被験者の一括確認

In [ ]:
# 複数被験者を一括で確認（Pages + Task Windows組み合わせ、画像保存）
# Bright: S01～S19, Dark: S101～S119, セグメント0～9

# 処理リストを作成
configs = []
for s in range(1, 20):
    for seg in range(0, 10):
        configs.append(("Bright", f"S{s:02d}", seg))
for s in range(101, 120):
    for seg in range(0, 10):
        configs.append(("Dark", f"S{s:03d}", seg))

print(f"🔄 処理開始: {len(configs)}件")
print(f"  Bright: S13～S19 × seg 0～9 (70件)")
print(f"  Dark: S101～S119 × seg 0～9 (190件)")
print("=" * 80)

success_count = 0
skip_count = 0
error_count = 0

for idx, (condition, subject_id, segment) in enumerate(configs, 1):
    print(f"\n[{idx}/{len(configs)}] 処理中: {condition} {subject_id} seg={segment}")
    print("-" * 80)
    
    try:
        # ファイルパスを構築
        pages_pattern = f"../../data/graphs/pages/{condition}/{subject_id}/{subject_id}_{segment}_with_emr/{subject_id}_{segment}_with_emr_page_*.png"
        task_windows_pattern = f"../../data/graphs/task_windows/{condition}/{subject_id}/{subject_id}_{segment}_*.png"
        
        print(f"  📂 ファイル検索中...")
        pages_files = sorted(glob.glob(pages_pattern))
        task_windows_files = sorted(glob.glob(task_windows_pattern))
        
        print(f"  ├─ Pages画像: {len(pages_files)}枚")
        print(f"  └─ Task Windows画像: {len(task_windows_files)}枚")
        
        if pages_files or task_windows_files:
            print(f"  🎨 画像生成中...")
            
            # 全体のレイアウトを計算
            n_pages = min(len(pages_files), 3)  # 最大3枚
            n_task_windows = len(task_windows_files)
            
            # 行数を決定（Pages行 + Task Windows行）
            n_rows = (1 if n_pages > 0 else 0) + (1 if n_task_windows > 0 else 0)
            
            if n_rows > 0:
                fig = plt.figure(figsize=(24, 8 * n_rows))
                
                current_row = 1
                
                # Pages画像を上段に横並び
                if n_pages > 0:
                    for i, img_path in enumerate(pages_files[:3]):
                        ax = plt.subplot(n_rows, 3, i + 1)
                        img = Image.open(img_path)
                        ax.imshow(img)
                        ax.axis('off')
                        ax.set_title(os.path.basename(img_path), fontsize=10)
                    current_row += 1
                
                # Task Windows画像を下段に表示
                if n_task_windows > 0:
                    for j, img_path in enumerate(task_windows_files):
                        # 下段は3列全体を使用
                        ax = plt.subplot(n_rows, 1, current_row + j)
                        img = Image.open(img_path)
                        ax.imshow(img)
                        ax.axis('off')
                        ax.set_title(os.path.basename(img_path), fontsize=10)
                
                plt.suptitle(f"{condition} {subject_id} seg={segment}", fontsize=14, y=0.98)
                plt.tight_layout(rect=[0, 0, 1, 0.97])
                
                # 画像を保存（被験者ごとのフォルダ）
                save_dir = f"../../data/graphs/page_task_combined/{condition}/{subject_id}"
                os.makedirs(save_dir, exist_ok=True)
                save_path = f"{save_dir}/{subject_id}_{segment}_combined.png"
                
                print(f"  💾 保存中...")
                fig.savefig(save_path, dpi=150, bbox_inches='tight', pad_inches=0.3)
                print(f"  ✅ 完了: {os.path.basename(save_path)}")
                
                plt.close()
                success_count += 1
        else:
            print(f"  ⚠️  画像が見つかりません（スキップ）")
            skip_count += 1
    
    except Exception as e:
        print(f"  ❌ エラー: {e}")
        error_count += 1
        continue

print("\n" + "=" * 80)
print("🎉 全処理完了")
print(f"  ✅ 成功: {success_count}件")
print(f"  ⚠️  スキップ: {skip_count}件")
print(f"  ❌ エラー: {error_count}件")
print(f"  📁 保存先: ../../data/graphs/page_task_combined/{{Bright|Dark}}/{{SubjectID}}/")
print("=" * 80)

🔄 処理開始: 5件
  Bright: S13～S19 × seg 0～9 (70件)
  Dark: S101～S119 × seg 0～9 (190件)

[1/5] 処理中: Dark S101 seg=5
--------------------------------------------------------------------------------
  📂 ファイル検索中...
  ├─ Pages画像: 3枚
  └─ Task Windows画像: 1枚
  🎨 画像生成中...
  💾 保存中...
  ✅ 完了: S101_5_combined.png

[2/5] 処理中: Dark S101 seg=6
--------------------------------------------------------------------------------
  📂 ファイル検索中...
  ├─ Pages画像: 3枚
  └─ Task Windows画像: 1枚
  🎨 画像生成中...
  💾 保存中...
  ✅ 完了: S101_6_combined.png

[3/5] 処理中: Dark S101 seg=7
--------------------------------------------------------------------------------
  📂 ファイル検索中...
  ├─ Pages画像: 3枚
  └─ Task Windows画像: 1枚
  🎨 画像生成中...
  💾 保存中...
  ✅ 完了: S101_7_combined.png

[4/5] 処理中: Dark S101 seg=8
--------------------------------------------------------------------------------
  📂 ファイル検索中...
  ├─ Pages画像: 3枚
  └─ Task Windows画像: 1枚
  🎨 画像生成中...
  💾 保存中...
  ✅ 完了: S101_8_combined.png

[5/5] 処理中: Dark S101 seg=9
------------------------